# 03. Выводы и бизнес-рекомендации

**Что делаем в этом ноутбуке:** превращаем находки из `02_analysis.ipynb` в три конкретные рекомендации, которые продуктовая команда может взять и реализовать. Никакой математики — только бизнес-язык и числа.

**Один вывод одной фразой:** ключевое окно для борьбы за повторную покупку — первые 30 дней; ключевая аудитория — клиенты с низким первым чеком; ключевая аномалия — декабрьские когорты, которые нужно обрабатывать отдельно.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

customers = pd.read_parquet('../data/customers_labeled.parquet')
retention = pd.read_parquet('../data/cohort_retention.parquet')
print(f'Клиентов в сегментации: {len(customers):,}')

## Краткое резюме находок

| Что нашли | Метрика | Что это значит для бизнеса |
|---|---|---|
| Главное падение retention — в первый месяц | с 100% до ~22% | Окно для action — первые 30 дней |
| Размер первого чека предсказывает возврат | t-тест, p ≪ 0.001 | Можно скорить клиентов сразу после первой покупки |
| Декабрьские когорты возвращаются хуже | viz на тепловой карте | Не тратить retention-бюджет на «подарочных» в январе |

---
## Рекомендация 1. Триггерная кампания реактивации в первые 14 дней — только для низкого чека

**Что делать:** запускать персональную email/push-кампанию с промокодом на 2-ю покупку, но **только** для клиентов, чей первый чек оказался ниже медианы. Окно — первые 14 дней после первой покупки.

**Почему именно так:**
- У сегмента «высокий чек» доля возврата и так высокая — экономить на них бюджет.
- У сегмента «низкий чек» возвратность ниже, и именно там потенциальный прирост retention имеет наибольшую ценность.
- Окно «первые 14 дней» выбрано потому, что критическое падение происходит в первый месяц — нужно успеть «достучаться» до того, как клиент окончательно «остыл».

In [ ]:
# Размер сегмента, на который ляжет кампания
low_check = customers[customers['check_segment'].str.startswith('Низкий')]
high_check = customers[customers['check_segment'].str.startswith('Высокий')]

lift_potential = high_check['returned'].mean() - low_check['returned'].mean()

print(f'Размер целевого сегмента (низкий чек):  {len(low_check):,} клиентов')
print(f'Текущая доля возврата в сегменте:        {low_check["returned"].mean()*100:.1f}%')
print(f'Доля возврата в сегменте «высокий чек»:  {high_check["returned"].mean()*100:.1f}%')
print(f'Разрыв (потенциал апсайда):              {lift_potential*100:.1f} п.п.')

**Как мерить успех кампании:**
- A/B-тест: 50% клиентов сегмента «низкий чек» получают триггерное письмо, 50% — нет.
- Главная метрика: доля совершивших 2-ю покупку в окне 30 дней.
- Контроль метрики: средний чек 2-й покупки (важно убедиться, что промо не каннибализирует выручку).
- Минимально детектируемый эффект (MDE): 2 п.п. — это амбициозная, но реалистичная цель для такой кампании.

---
## Рекомендация 2. Декабрьские клиенты — отдельная воронка

**Что делать:** клиентов, у которых первая покупка приходится на декабрь, исключать из стандартных январских и февральских кампаний реактивации. Вместо этого делать им осторожную «приветственную» серию: контент о категориях, которые они не покупали, без промо-кодов.

**Почему:**
- Декабрьские когорты имеют специфический мотив покупки — подарок, не для себя.
- Стандартная кампания «вернись с промо на свой любимый товар» им неактуальна — они купили подарок, а не для себя.
- Промо-коды на этот сегмент — это в основном выкинутые деньги.

**Цифра:** на тепловой карте видно, что декабрьские когорты в M+1 (январь) возвращаются хуже сравнимых месяцев.

In [ ]:
# Сравним retention M+1 у декабрьских когорт vs остальных
december_mask = retention.index.month == 12
dec_retention_m1 = retention.loc[december_mask, 1].mean()
other_retention_m1 = retention.loc[~december_mask, 1].mean()

print(f'Retention M+1 у декабрьских когорт:  {dec_retention_m1:.1f}%')
print(f'Retention M+1 у остальных когорт:    {other_retention_m1:.1f}%')
print(f'Разница:                              {other_retention_m1 - dec_retention_m1:.1f} п.п.')

---
## Рекомендация 3. Первый чек — ранний скоринговый сигнал

**Что делать:** добавить в продуктовую аналитику метрику «первый чек выше/ниже медианы» как ранний предиктор LTV. Использовать в дашбордах онбординга и в моделях прогноза LTV (даже без сложного ML — просто как фичу).

**Почему:**
- Уже на следующий день после первой покупки мы знаем, в какой группе клиент.
- Это даёт CRM-команде грубую сегментацию «бесплатно», без накопления истории.

**Чего не делать:** ни в коем случае не строить рекламную/продуктовую логику в духе «клиент с низким чеком = плохой клиент, не показываем дорогие товары». Это путь в самоисполняющееся пророчество. Сегментация полезна только для **догоняющих** активностей (реактивация), но не для дискриминации в продукте.

---
## Что я НЕ сделал и почему

На стажёрской позиции честность про ограничения важнее показного «я всё могу». Список того, что осталось за кадром:

1. **Не построил модель прогноза возврата.** Сознательно: t-тест и сегментация по медиане отвечают на бизнес-вопрос, а ML добавил бы сложности без понятного бизнес-выигрыша. Если задача потребует — модель строится поверх той же выборки.
2. **Не доказал причинность.** t-тест — это про корреляцию. Чтобы доказать, что увеличение первого чека реально *вызывает* рост retention, нужен A/B-тест с воздействием на первый чек (например, через бандлы при онбординге).
3. **Не сделал когортный анализ по другим срезам** (категория товара, страна, источник трафика — последнего в датасете и нет). При расширении проекта это первое, что нужно добавить.
4. **Не учёл уникальность B2B/оптовых клиентов.** В Online Retail II большая доля выручки — оптовые покупатели. Их поведение не такое же, как у B2C. Для маркетплейса вроде Маркета этот эффект был бы другим.

**Что я сделал бы дальше при наличии времени:**
- RFM-сегментация (Recency / Frequency / Monetary) — стандартный следующий шаг.
- Расчёт LTV по сегментам и сравнение unit-экономики.
- Симуляция A/B-теста: какой минимальный размер выборки нужен, чтобы поймать эффект 2 п.п. на retention при текущей вариативности.

## Финальная фраза одной строкой

> **Самый дешёвый способ повысить retention — не пытаться удержать всех, а вычислить тех, кто и так не уйдёт, и не тратить на них бюджет. Размер первого чека — простой и ранний сигнал, который позволяет это сделать.**